In [1]:
%%capture

import warnings
warnings.filterwarnings("ignore")

import altair as alt
import branca.colormap as cm
import folium
import gcsfs
import pandas as pd

import calitp_data_analysis.magics
import gt_extras as gte

from great_tables import GT, md

import chart_utils_for_operators as chart_utils
import prep_operator_data
import report_utils
import _color_palette
PREDICTIONS_GCS = "gs://calitp-analytics-data/data-analyses/rt_predictions/"

alt.data_transformers.enable("vegafusion")

one_month = pd.to_datetime("2026-02-01")
date_format = "%b %Y" # gtfs_digest/_new_operator_report_utils.py
one_month_formatted = one_month.strftime(date_format)

In [2]:
operator_cols = ["tu_name", "day_type"]

schedule_cols = [
    "daily_trips", "daily_service_hours", "n_routes", "n_shapes",
    "n_stops", "num_stop_times", "daily_arrivals", 'n_days_schedule_and_rt'
]

tu_cols = ['tu_messages_per_minute', 'n_tu_trips', 'pct_tu_trips', 'pct_tu_routes'] #'daily_tu_trips',

# split prediction cols into 2 separate tables so comparison is clearer
tu_prediction_cols1 = ["bus_catch_likelihood", "pct_tu_complete_minutes", "p50", "n_predictions"]
tu_prediction_cols2 = [
    "p25", "p75", "iqr",
    "prediction_padding_minutes", "avg_prediction_spread_minutes"
]

In [3]:
equans = "Marin EQUANS Trip Update"
swiftly = "Marin Swiftly Trip Update"
mtc = "Bay Area 511 Marin TripUpdates"

df = pd.read_parquet(
    f"{PREDICTIONS_GCS}monthly_operator_summary_marin.parquet",
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("tu_name", "in", [equans, swiftly, mtc])]],
).pipe(prep_operator_data.merge_in_operator_percentiles)

In [4]:
# Set variables for color bars used across maps, route dropdown, and great tables
PREDICTION_ERROR_COLORS =list(_color_palette.PREDICTION_ERROR_COLOR_PALETTE.values())
PREDICTION_ERROR_INDEX = [-5, -3, -1, 1, 3, 5]
PREDICTION_ERROR_LEGEND_CAPTION = "minutes (negative = late; positive = early)"

POS_BAR_COLOR = _color_palette.get_color("blueberry")
NEG_BAR_COLOR = _color_palette.get_color("vivid_cerise")

## Schedule + RT Summary Stats

In [ ]:
schedule_table = (
    GT(df[["schedule_name"] + operator_cols + schedule_cols])
    .cols_label(
        daily_trips = "Daily Trips",
        daily_service_hours = "Daily Service Hours",
        n_routes = "# Routes",
        n_shapes = "# Shapes",
        n_stops = "# Stops",
        num_stop_times = "Total Scheduled Arrivals",
        daily_arrivals = "Daily Scheduled Arrivals",    
        n_days_schedule_and_rt = "# days with both RT",
    ).fmt_integer(
        columns = [
            "daily_trips", "n_routes", "n_shapes", "n_stops", 
            "num_stop_times", "daily_arrivals", "n_days_schedule_and_rt"]
    ).fmt_number(
        columns = ["daily_service_hours"], decimals=1
    ).tab_spanner(
        label="Schedule",
        columns = schedule_cols
    ).tab_header(
        title = "Schedule + RT Summary Metrics", 
        subtitle = f"{one_month_formatted}"
    )
)

chart_utils.format_great_table(schedule_table, day_type_grouping = False).tab_stub(
    groupname_col="tu_name", rowname_col = "day_type")

## General RT Metrics

<span style="color:#4477aa">**Update Availability Goal 1:** 2+ vehicle positions or trip updates messages per minute.</span>

<span style="color:#4477aa">**Update Availability Goal 2:** 100% routes are covered by RT, and 75%+ of trips have RT.</span>

In [ ]:
rt_table1 = (
    GT(df[["schedule_name"] + operator_cols + ["n_trips"] + tu_cols])
    .cols_label(
        schedule_name = "Schedule",
        n_trips = "# Scheduled Trips",
        n_tu_trips = "# Trips (Trip Updates)",
        tu_messages_per_minute = "Messages per Minute",
        pct_tu_trips = "% Trips",
        pct_tu_routes = "% Routes",
    ).fmt_number(
        columns = ["tu_messages_per_minute"], decimals=1
    ).fmt_percent(
        columns=["pct_tu_trips", "pct_tu_routes"], decimals=1
    ).fmt_integer(columns = ["n_trips", "n_tu_trips"]
    ).tab_stub(rowname_col="day_type", groupname_col="tu_name")
    .cols_hide(columns=["tu_name"])
)

chart_utils.format_great_table(rt_table1, day_type_grouping = False).pipe(
    gte.gt_color_box, 
    columns=["tu_messages_per_minute"], 
    palette="Blues",
    domain=[1, 3]
).pipe(
    gte.gt_hulk_col_numeric, 
    columns=["pct_tu_trips", "pct_tu_routes"],
    palette=[_color_palette.get_color("light_goldenrod"), 
             _color_palette.get_color("pastel_peppermint")], 
    domain=[0, 1],
    na_color="black", # with opacity at 0.1, this is basically gray
    alpha=0.1
)

In [ ]:
rt_table2 = (
    GT(df[operator_cols + tu_prediction_cols1])
    .cols_label(
        pct_tu_complete_minutes = "% Minutes with 2+ Predictions",
        bus_catch_likelihood = "Bus Catch Likelihood (Early + On-time)",
        p50 = "Prediction Error", 
        n_predictions = "# Predictions",
    ).fmt_percent(columns=["bus_catch_likelihood", "pct_tu_complete_minutes"], decimals=1)
    .fmt_number(columns=["p50"], decimals=1)
    .fmt_integer(columns=["n_predictions"])
    .tab_header(
        title = f"Trip Update Prediction Accuracy Metrics", 
        subtitle = "units are in minutes")
).cols_hide(
    columns=["tu_name"]
).pipe(
    chart_utils.format_great_table
).tab_stub(
    rowname_col="day_type", groupname_col="tu_name"
)

rt_table2.pipe(
    gte.gt_hulk_col_numeric, 
    columns=["bus_catch_likelihood", "pct_tu_complete_minutes"],
    palette=[_color_palette.get_color("light_goldenrod"), 
             _color_palette.get_color("pastel_peppermint")],
    domain=[0, 1],
    alpha=0.1
).pipe(
    gte.gt_color_box, 
    columns=["p50"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]  
).cols_move_to_end(columns=["n_predictions"])

Not colored in main report, which is individual feeds.
Here, since all feeds are put together, color them so we can see how they compare.

* Variability (IQR) is ranked across the 3 feeds, so it's more interpretable as a comparison.
* Prediction Spread, prediction padding


In [ ]:
rt_table3 = (
    GT(df[operator_cols + ["p50"] + tu_prediction_cols2])
    .cols_label(
        p50 = "Prediction Error", 
        avg_prediction_spread_minutes = "Prediction Spread / Wobble",
        prediction_padding_minutes = "Prediction Padding",
        iqr = "Variability"
    )
    .fmt_number(columns=["p50", "avg_prediction_spread_minutes", "prediction_padding_minutes"], decimals=1)
    .tab_header(title = f"Trip Update Prediction Accuracy Metrics", 
                subtitle = "units are in minutes")
).pipe(
    chart_utils.format_great_table
).tab_stub(
    rowname_col="day_type", groupname_col="tu_name"
)


rt_table3.pipe(
    gte.gt_color_box, 
    columns=["p50"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]  
).pipe(
    gte.gt_plt_dumbbell,
    col1='p25',
    col2='p75',
    label = "IQR",
    num_decimals=1,
    col1_color=_color_palette.get_color("valentino"),
    col2_color=_color_palette.get_color("lizard_green"),
    width=100, height=50, # check these each time
    font_size=8
).pipe(
    gte.gt_color_box, 
    columns=["iqr"], 
    palette="YlOrRd"
).pipe(
    gte.gt_color_box,
    columns=["prediction_padding_minutes", "avg_prediction_spread_minutes"],
    palette="BuPu"
)

## Prediction Error Percentiles

### Distribution of Prediction Errors

The 50th percentile is the typical or median rider experience, and it can show that, on average, this transit agency is roughly on-time. 
* If the 10th percentile is fairly close to the 50th percentile, it means that the transit agency is consistent and reliable in its predictions. 
* Extreme values for the 10th percentile would indicate that predictions fluctuate, or, are somewhat unreliable.
* Steeper lines indicate fairly reliable predictions for the rider.

In [12]:
decile_cols = [
    "month_first_day", "day_type",
    "schedule_name", "tu_name",
    'pos_prediction_error_sec_array', 'pos_prediction_error_sec_percentile_array', 
    'neg_prediction_error_sec_array', 'prediction_error_sec_percentile_array'
]

operator_deciles_df = pd.read_parquet(
    f"{PREDICTIONS_GCS}monthly_operator_summary_marin.parquet",
    filesystem=gcsfs.GCSFileSystem(),
    filters = [[("tu_name", "in", [equans, swiftly, mtc])]],
    columns = decile_cols
).pipe(
    prep_operator_data.operator_deciles_for_chart
)

In [45]:
def percentile_chart_for_day_type(
    operator_deciles_df: pd.DataFrame, 
    day_type: str
) -> alt.Chart:
    """
    Create a percentile chart for each day_type + 3 feeds in the legend.
    """
    subset_df = operator_deciles_df[operator_deciles_df.day_type==day_type]
    
    percentile_chart = chart_utils.fig5and6_prediction_error_plots(
        subset_df, color_col="tu_name"
    ).properties(title=day_type, height=200, width=200)

    return percentile_chart

In [49]:
day_types = ["Weekday", "Saturday", "Sunday"]

chart_list = [
    percentile_chart_for_day_type(operator_deciles_df, d) 
    for d in day_types
]

alt.hconcat(*chart_list).properties(
    title={
        "text": "Prediction Error Percentiles - Feed Comparison",
        "subtitle": "use legend to select feed"
    }
)

alt.HConcatChart(...)

### Accuracy Loss 
Ratio of the 10th to 50th percentiles 

* Newmark's paper on a small sample of transit agencies suggests that the positive prediction errors typically have ratios of 4.
* Late predictions (negative prediction errors) have ratios around 3.
* Steeper lines = less accuracy loss = better
   * y-axis is percentile (moving from 10th to 50th percentile is moving from upwards on y-axis)
   * x-axis is error (smaller change along x-axis is less accuracy loss).
   * less accuracy loss = less change along x-axis, since change along y-axis is constant (10 to 50) = steeper (unintuitive to the normal interpretation of slope!)

In [52]:
operator_cols = ["tu_name", "day_type"]
percentile_chart_cols = [
    "pos_p10", "pos_p50", "pos_error_ratio", 
    "neg_p10", "neg_p50", "neg_error_ratio"
]

mini_df = df[operator_cols + percentile_chart_cols]

# convert the 10th, 50th percentile columns to minutes
seconds_cols = ["pos_p10", "pos_p50", "neg_p10", "neg_p50"]
mini_df[seconds_cols] = mini_df[seconds_cols].divide(60).round(2)

In [58]:
mini_p10_p50_table = (
    GT(mini_df)
    .cols_label(
        pos_p10 = "10th percentile ",
        neg_p10 = "10th percentile",
        pos_p50 = "50th percentile",
        neg_p50 = "50th percentile",
        pos_error_ratio = "Accuracy Loss", 
        neg_error_ratio = "Accuracy Loss",
    ).fmt_number(
        columns=["pos_error_ratio", "neg_error_ratio"], decimals=1
    ).tab_spanner(
        label="Early Predictions (Positive Prediction Error)",
        columns=["pos_p10", "pos_p50", "pos_error_ratio"]
    ).tab_spanner(
        label="Late Predictions (Negative Prediction Error)",
        columns = ["neg_p10", "neg_p50", "neg_error_ratio"]
    )
    .tab_header(
        title = "Accuracy Loss = Ratio of 10th to 50th percentile error", 
        subtitle = "units are in minutes (lower = less accuracy loss)"
    )
).pipe(
    gte.gt_color_box, 
    columns=["pos_p10", "pos_p50", "neg_p10", "neg_p50"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]
).pipe(
    gte.gt_color_box,
    columns=["pos_error_ratio", "neg_error_ratio"],
    palette="YlOrRd"
)

chart_utils.format_great_table(mini_p10_p50_table).pipe(
    chart_utils.format_great_table, 
    day_type_grouping=False
).tab_stub(
    rowname_col="day_type", groupname_col="tu_name"
)

GT(_tbl_data=                          tu_name  day_type  pos_p10  pos_p50  \
0  Bay Area 511 Marin TripUpdates   Weekday     5.05     1.58   
1  Bay Area 511 Marin TripUpdates  Saturday     4.37     1.37   
2  Bay Area 511 Marin TripUpdates    Sunday     4.32     1.30   
3        Marin EQUANS Trip Update   Weekday     7.71     2.97   
4        Marin EQUANS Trip Update  Saturday     7.57     2.93   
5        Marin EQUANS Trip Update    Sunday     7.33     2.83   
6       Marin Swiftly Trip Update   Weekday     5.03     1.58   
7       Marin Swiftly Trip Update  Saturday     4.35     1.36   
8       Marin Swiftly Trip Update    Sunday     4.30     1.28   

   pos_error_ratio  neg_p10  neg_p50  neg_error_ratio  
0              3.2    -2.38    -0.60              4.0  
1              3.2    -2.58    -0.65              4.0  
2              3.3    -2.80    -0.68              4.1  
3              2.6    -4.21    -0.67              6.3  
4              2.6    -4.02    -0.70              5.7  
5              2.6    -5.06    -0.66              7.6  
6              3.2    -2.37    -0.58              4.1  
7              3.2    -2.57    -0.65              3.9  
8              3.3    -2.78    -0.67              4.1  , _body=<great_tables._gt_data.Body object at 0x7a8ccdf39390>, _boxhead=Boxhead([ColInfo(var='tu_name', type=<ColInfoTypeEnum.row_group: 3>, column_label='tu_name', column_align='center', column_width=None), ColInfo(var='day_type', type=<ColInfoTypeEnum.stub: 2>, column_label='day_type', column_align='center', column_width=None), ColInfo(var='pos_p10', type=<ColInfoTypeEnum.default: 1>, column_label='10th percentile ', column_align='center', column_width=None), ColInfo(var='pos_p50', type=<ColInfoTypeEnum.default: 1>, column_label='50th percentile', column_align='center', column_width=None), ColInfo(var='pos_error_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='Accuracy Loss', column_align='center', column_width=None), ColInfo(var='neg_p10', type=<ColInfoTypeEnum.default: 1>, column_label='10th percentile', column_align='center', column_width=None), ColInfo(var='neg_p50', type=<ColInfoTypeEnum.default: 1>, column_label='50th percentile', column_align='center', column_width=None), ColInfo(var='neg_error_ratio', type=<ColInfoTypeEnum.default: 1>, column_label='Accuracy Loss', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7a8cb7729250>, _spanners=Spanners([SpannerInfo(spanner_id='Early Predictions (Positive Prediction Error)', spanner_level=0, spanner_label='Early Predictions (Positive Prediction Error)', spanner_units=None, spanner_pattern=None, vars=['pos_p10', 'pos_p50', 'pos_error_ratio'], built=None), SpannerInfo(spanner_id='Late Predictions (Negative Prediction Error)', spanner_level=0, spanner_label='Late Predictions (Negative Prediction Error)', spanner_units=None, spanner_pattern=None, vars=['neg_p10', 'neg_p50', 'neg_error_ratio'], built=None)]), _heading=Heading(title='Accuracy Loss = Ratio of 10th to 50th percentile error', subtitle='units are in minutes (lower = less accuracy loss)', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7a8cb77a4c90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7a8ccefae350>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7a8cb77a6690>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7a8ccdf1eb50>, <great_tables._gt_data.FormatInfo object at 0x7a8cb7797850>, <great_tables._gt_data.FormatInfo object at 0x7a8cb7796810>, <great_tables._gt_data.FormatInfo object at 0x7a8cb7797510>, <great_tables._gt_data.FormatInfo object at 0x7a8cb7797cd0>, <great_tables._gt_data.FormatInfo object at 0x7a8cb7797790>, <great_tables._gt_data.FormatInfo object at 0x7a8cb7797990>, <great_tables._gt_data.FormatInfo object at 0x7a8cb7797950>, <great_tables._gt_data.FormatInfo object at 0x7a8cb7794ad0>, <great_tables._gt_data.FormatInfo obje

## Route Summary

Prediction accuracy varies by routes. The routes shown at the top have high variability (high IQRs).

* **High variability = high IQRs**: local traffic conditions mat confound the prediction algorithm. For these routes, a focus on improving service reliability through additional infrastructure (signal priority, bus lanes), or other transit planning and policies could be explored.

* **Negative 25th percentiles**: riders miss the bus (late predictions). These routes may benefit from service reliability improvements for riders.

   *Interpretation*: A value of -5 means that one quarter of riders miss the bus by 5 minutes.

* **scaled IQR**: IQR adjusted so predictions closer to the bus arrival are weighted more. Predictions 5 minutes out are more important than predictions 25 minutes out.

In [ ]:
# merge these together
route_df = "fct_monthly_schedule_rt_route_direction_summary_marin"
route_gdf = "fct_monthly_routes_marin"

In [ ]:

route_iqr_df = report_utils.import_route_df(
    filters = [[
        ("tu_name", "==", name),
        ("month_first_day", "==", one_month),
        ("day_type", "==", "Weekday")
    ]],
    columns= [
        "route_dir_name", 
        "avg_prediction_error_minutes",
        "n_predictions",
        "p25", "p75",
        "iqr",s
        "scaled_p25", "scaled_p75",
        # does putting IQR help in interpretation?
    ]
).sort_values("iqr", ascending=False)

In [ ]:
route_iqr_table = (
    GT(route_iqr_df)
    .cols_label(
        route_dir_name = "Route-Direction",
        n_predictions = "# Predictions",
        iqr = "Variability",
        avg_prediction_error_minutes = "Prediction Error (minutes)",
    ).fmt_integer(["n_predictions"])
    .tab_header(
        title = "Route Summary Metrics",
        subtitle = md(
            """
            High IQR = variability -> focus on service reliability through transit planning and policies. 
            Variability could be due to local traffic conditions. 
            Negative 25th percentiles = riders miss bus."""
        )
    ).pipe(chart_utils.format_great_table, day_type_grouping=False)
)

route_iqr_table.pipe(
    gte.gt_plt_dumbbell,
    col1='p25',
    col2='p75',
    label = "IQR (minutes)",
    num_decimals=1,
    width=200, height=50,
    col1_color=_color_palette.get_color("valentino"),
    col2_color=_color_palette.get_color("lizard_green"),
    font_size=8
).pipe(
    gte.gt_plt_dumbbell,
    col1="scaled_p25",
    col2='scaled_p75',
    label='scaled IQR',
    num_decimals=3,
    width=200, height=50,
    col1_color=_color_palette.get_color("valentino"),
    col2_color=_color_palette.get_color("lizard_green"),
    font_size=8
).pipe(
    gte.gt_color_box, 
    columns=["avg_prediction_error_minutes"], 
    palette=PREDICTION_ERROR_COLORS,
    domain=[-5, 5]
).pipe(
    gte.gt_color_box, 
    columns=["iqr"], 
    palette="YlOrRd",
).cols_width(
    cases={
        "avg_prediction_error_minutes": "10%",
        "n_predictions": "10%",
        "iqr": "10%"
    }
).cols_align("center").cols_align(
    "left", columns = "route_dir_name"
).cols_move_to_end(columns=["n_predictions"])